# E-Commerce Marketing Analytics & Customer Segmentation

### End-to-End Data Analyst Portfolio Project

**Tools:** Python • Pandas • NumPy • SQL • Power BI



In [ ]:
import pandas as pd
import numpy as np

base_path = "../data/e-commerce-customer-data/"

customers = pd.read_csv(base_path + "customers.csv")
sessions = pd.read_csv(base_path + "sessions.csv")
transactions = pd.read_csv(base_path + "transactions.csv")
campaigns = pd.read_csv(base_path + "marketing_campaigns.csv")
geo = pd.read_csv(base_path + "geo_data.csv")

print("Raw data loaded successfully!")
print("Customers:", len(customers))
print("Sessions:", len(sessions))
print("Transactions:", len(transactions))
print("Campaigns:", len(campaigns))
print("Geo:", len(geo))

## 1.Data Cleaning and Validation

In [ ]:
import pandas as pd
import numpy as np

# Create fresh copies from the original raw data
customers_clean = customers.copy()
sessions_clean = sessions.copy()
transactions_clean = transactions.copy()
campaigns_clean = campaigns.copy()
geo_clean = geo.copy()

#customers
customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"]
)

#sessions
sessions_clean["session_timestamp"] = pd.to_datetime(
    sessions_clean["session_timestamp"]
)

sessions_clean["traffic_source"] = (
    sessions_clean["traffic_source"]
    .astype(str)
    .str.strip()
    .str.lower()
)

sessions_clean["device_type"] = (
    sessions_clean["device_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

sessions_clean["bounce_flag"] = (
    sessions_clean["bounce_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

#transactions
transactions_clean["transaction_timestamp"] = pd.to_datetime(
    transactions_clean["transaction_timestamp"]
)

transactions_clean["order_value"] = (
    transactions_clean["order_value"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

transactions_clean["discount_applied"] = (
    transactions_clean["discount_applied"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

transactions_clean["high_value_flag"] = (
    transactions_clean["high_value_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "no": 0
    })
)

#campaigns
campaigns_clean["campaign_budget"] = (
    campaigns_clean["campaign_budget"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

campaigns_clean["start_date"] = pd.to_datetime(
    campaigns_clean["start_date"]
)

campaigns_clean["end_date"] = pd.to_datetime(
    campaigns_clean["end_date"]
)


customers_clean = customers_clean.drop(
    columns=[c for c in customers_clean.columns if c.startswith("noise_")]
)

transactions_clean = transactions_clean.drop(
    columns=[c for c in transactions_clean.columns if c.startswith("noise_")]
)

print("CUSTOMERS:", customers_clean.shape)
print("SESSIONS:", sessions_clean.shape)
print("TRANSACTIONS:", transactions_clean.shape)
print("CAMPAIGNS:", campaigns_clean.shape)
print("GEO:", geo_clean.shape)

print("\nMissing values:")
print("Customers:", customers_clean.isna().sum().sum())
print("Sessions:", sessions_clean.isna().sum().sum())
print("Transactions:", transactions_clean.isna().sum().sum())
print("Campaigns:", campaigns_clean.isna().sum().sum())

print("\nCleaning completed.")

In [ ]:
# Find missing values in each cleaned table

print("CUSTOMERS")
print(customers_clean.isna().sum()[customers_clean.isna().sum() > 0])

print("\nSESSIONS")
print(sessions_clean.isna().sum()[sessions_clean.isna().sum() > 0])

print("\nTRANSACTIONS")
print(transactions_clean.isna().sum()[transactions_clean.isna().sum() > 0])

In [ ]:
#handling missing values

customers_clean["email_open_rate_missing"] = (
    customers_clean["email_open_rate"].isna().astype(int)
)

sessions_clean["pages_viewed_missing"] = (
    sessions_clean["pages_viewed"].isna().astype(int)
)

# Impute using median
email_median = customers_clean["email_open_rate"].median()
pages_median = sessions_clean["pages_viewed"].median()

customers_clean["email_open_rate"] = (
    customers_clean["email_open_rate"].fillna(email_median)
)

sessions_clean["pages_viewed"] = (
    sessions_clean["pages_viewed"].fillna(pages_median)
)

print("Email open rate median used:", email_median)
print("Pages viewed median used:", pages_median)

print("\nRemaining missing values:")
print("Customers:", customers_clean.isna().sum().sum())
print("Sessions:", sessions_clean.isna().sum().sum())

## 2. Customer Acquisition Analysis

In [ ]:
# Sort sessions chronologically
sessions_sorted = sessions_clean.sort_values(
    ["customer_id", "session_timestamp"]
)

# First observed session for each customer
first_sessions = (
    sessions_sorted
    .drop_duplicates("customer_id", keep="first")
)

# Keep acquisition information
customer_acquisition = first_sessions[
    [
        "customer_id",
        "session_timestamp",
        "traffic_source",
        "campaign_id"
    ]
].copy()

# Rename columns
customer_acquisition = customer_acquisition.rename(
    columns={
        "session_timestamp": "acquisition_date",
        "traffic_source": "acquisition_channel",
        "campaign_id": "acquisition_campaign"
    }
)

# Add campaign information
customer_acquisition = customer_acquisition.merge(
    campaigns_clean[
        [
            "campaign_id",
            "campaign_type",
            "campaign_budget"
        ]
    ],
    left_on="acquisition_campaign",
    right_on="campaign_id",
    how="left"
)

# Remove duplicate campaign ID
customer_acquisition = customer_acquisition.drop(
    columns=["campaign_id"]
)

# Rename campaign columns
customer_acquisition = customer_acquisition.rename(
    columns={
        "campaign_type": "acquisition_campaign_type",
        "campaign_budget": "acquisition_campaign_budget"
    }
)

# Check result
print("Shape:", customer_acquisition.shape)
print(
    "Unique customers:",
    customer_acquisition["customer_id"].nunique()
)

print("\nAcquisition channels:")
print(
    customer_acquisition["acquisition_channel"]
    .value_counts()
)

print("\nMissing values:")
print(customer_acquisition.isna().sum())

print("\nSample:")
print(customer_acquisition.head())

In [ ]:
#customer-vele business metrics

# Aggregate transactions to customer level
customer_transactions = (
    transactions_clean
    .groupby("customer_id")
    .agg(
        total_orders=("transaction_id", "count"),
        total_revenue=("order_value", "sum"),
        avg_order_value=("order_value", "mean"),
        total_items=("items_count", "sum"),
        discounted_orders=("discount_applied", "sum"),
        high_value_orders=("high_value_flag", "sum"),
        first_purchase_date=("transaction_timestamp", "min"),
        last_purchase_date=("transaction_timestamp", "max")
    )
    .reset_index()
)

# Merge acquisition information
customer_metrics = customer_acquisition.merge(
    customer_transactions,
    on="customer_id",
    how="left"
)

# Customers with no transactions
customer_metrics["total_orders"] = (
    customer_metrics["total_orders"].fillna(0)
)

customer_metrics["total_revenue"] = (
    customer_metrics["total_revenue"].fillna(0)
)

customer_metrics["avg_order_value"] = (
    customer_metrics["avg_order_value"].fillna(0)
)

customer_metrics["total_items"] = (
    customer_metrics["total_items"].fillna(0)
)

customer_metrics["discounted_orders"] = (
    customer_metrics["discounted_orders"].fillna(0)
)

customer_metrics["high_value_orders"] = (
    customer_metrics["high_value_orders"].fillna(0)
)

# Derived customer-level metrics
customer_metrics["is_customer"] = (
    customer_metrics["total_orders"] > 0
).astype(int)

customer_metrics["is_repeat_customer"] = (
    customer_metrics["total_orders"] >= 2
).astype(int)

customer_metrics["discount_order_rate"] = np.where(
    customer_metrics["total_orders"] > 0,
    customer_metrics["discounted_orders"]
    / customer_metrics["total_orders"],
    0
)

customer_metrics["high_value_order_rate"] = np.where(
    customer_metrics["total_orders"] > 0,
    customer_metrics["high_value_orders"]
    / customer_metrics["total_orders"],
    0
)


print("Customer metrics shape:", customer_metrics.shape)

print(
    "\nCustomers with at least one purchase:",
    customer_metrics["is_customer"].sum()
)

print(
    "Repeat customers:",
    customer_metrics["is_repeat_customer"].sum()
)

print(
    "Overall revenue:",
    round(customer_metrics["total_revenue"].sum(), 2)
)

print(
    "\nAverage order value:",
    round(
        transactions_clean["order_value"].mean(),
        2
    )
)

print("\nSample:")
print(customer_metrics.head())

In [ ]:
#acquisation channel performance

channel_performance = (
    customer_metrics
    .groupby("acquisition_channel")
    .agg(
        acquired_customers=("customer_id", "nunique"),
        purchasing_customers=("is_customer", "sum"),
        repeat_customers=("is_repeat_customer", "sum"),
        total_revenue=("total_revenue", "sum"),
        total_orders=("total_orders", "sum"),
        total_items=("total_items", "sum")
    )
    .reset_index()
)

# Derived metrics
channel_performance["purchase_conversion_rate"] = (
    channel_performance["purchasing_customers"]
    / channel_performance["acquired_customers"]
)

channel_performance["repeat_customer_rate"] = (
    channel_performance["repeat_customers"]
    / channel_performance["purchasing_customers"]
)

channel_performance["revenue_per_acquired_customer"] = (
    channel_performance["total_revenue"]
    / channel_performance["acquired_customers"]
)

channel_performance["average_order_value"] = (
    channel_performance["total_revenue"]
    / channel_performance["total_orders"]
)

channel_performance["orders_per_customer"] = (
    channel_performance["total_orders"]
    / channel_performance["purchasing_customers"]
)

# Sort by revenue
channel_performance = channel_performance.sort_values(
    "total_revenue",
    ascending=False
)

print(channel_performance.to_string(index=False))

## 3. Campaign Performance Analysis

In [ ]:
campaign_performance = (
    customer_metrics
    .groupby(
        [
            "acquisition_campaign",
            "acquisition_campaign_type"
        ]
    )
    .agg(
        acquired_customers=("customer_id", "nunique"),
        purchasing_customers=("is_customer", "sum"),
        repeat_customers=("is_repeat_customer", "sum"),
        total_revenue=("total_revenue", "sum"),
        total_orders=("total_orders", "sum")
    )
    .reset_index()
)

# Attach campaign budget once per campaign
campaign_budgets = campaigns_clean[
    [
        "campaign_id",
        "campaign_budget",
        "region_target"
    ]
].copy()

campaign_performance = campaign_performance.merge(
    campaign_budgets,
    left_on="acquisition_campaign",
    right_on="campaign_id",
    how="left"
)

campaign_performance = campaign_performance.drop(
    columns=["campaign_id"]
)

# Derived metrics
campaign_performance["conversion_rate"] = (
    campaign_performance["purchasing_customers"]
    / campaign_performance["acquired_customers"]
)

campaign_performance["repeat_rate"] = (
    campaign_performance["repeat_customers"]
    / campaign_performance["purchasing_customers"]
)

campaign_performance["revenue_per_customer"] = (
    campaign_performance["total_revenue"]
    / campaign_performance["acquired_customers"]
)

campaign_performance["budget_based_cac"] = (
    campaign_performance["campaign_budget"]
    / campaign_performance["acquired_customers"]
)

campaign_performance["budget_based_roas"] = (
    campaign_performance["total_revenue"]
    / campaign_performance["campaign_budget"]
)

# Rank campaigns
campaign_performance["revenue_rank"] = (
    campaign_performance["total_revenue"]
    .rank(method="dense", ascending=False)
)

campaign_performance["roas_rank"] = (
    campaign_performance["budget_based_roas"]
    .rank(method="dense", ascending=False)
)

# Sort by revenue
campaign_performance = campaign_performance.sort_values(
    "total_revenue",
    ascending=False
)

print("Campaign performance shape:", campaign_performance.shape)

print("\nTop 10 campaigns by revenue:")
print(
    campaign_performance[
        [
            "acquisition_campaign",
            "acquisition_campaign_type",
            "acquired_customers",
            "purchasing_customers",
            "total_revenue",
            "campaign_budget",
            "budget_based_cac",
            "budget_based_roas"
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
# marketing funnel by acquisation channel

# Add acquisition channel to every session
sessions_with_acquisition = sessions_clean.merge(
    customer_acquisition[
        ["customer_id", "acquisition_channel"]
    ],
    on="customer_id",
    how="left"
)

# Create engagement indicators
sessions_with_acquisition["engaged_session"] = (
    (sessions_with_acquisition["bounce_flag"] == 0) |
    (sessions_with_acquisition["pages_viewed"] > 1) |
    (sessions_with_acquisition["cart_additions"] > 0)
).astype(int)

sessions_with_acquisition["cart_session"] = (
    sessions_with_acquisition["cart_additions"] > 0
).astype(int)

# Aggregate session-level funnel metrics
funnel_sessions = (
    sessions_with_acquisition
    .groupby("acquisition_channel")
    .agg(
        total_sessions=("session_id", "count"),
        engaged_sessions=("engaged_session", "sum"),
        cart_sessions=("cart_session", "sum"),
        total_cart_additions=("cart_additions", "sum"),
        bounced_sessions=("bounce_flag", "sum")
    )
    .reset_index()
)

# Customer-level metrics
funnel_customers = (
    customer_metrics
    .groupby("acquisition_channel")
    .agg(
        acquired_customers=("customer_id", "nunique"),
        purchasing_customers=("is_customer", "sum"),
        repeat_customers=("is_repeat_customer", "sum")
    )
    .reset_index()
)

marketing_funnel = funnel_customers.merge(
    funnel_sessions,
    on="acquisition_channel",
    how="left"
)

# Derived metrics
marketing_funnel["engagement_rate"] = (
    marketing_funnel["engaged_sessions"]
    / marketing_funnel["total_sessions"]
)

marketing_funnel["cart_session_rate"] = (
    marketing_funnel["cart_sessions"]
    / marketing_funnel["total_sessions"]
)

marketing_funnel["bounce_rate"] = (
    marketing_funnel["bounced_sessions"]
    / marketing_funnel["total_sessions"]
)

marketing_funnel["purchase_rate"] = (
    marketing_funnel["purchasing_customers"]
    / marketing_funnel["acquired_customers"]
)

marketing_funnel["sessions_per_customer"] = (
    marketing_funnel["total_sessions"]
    / marketing_funnel["acquired_customers"]
)

print(marketing_funnel.to_string(index=False))

## 4. Customer Segmentation & RFM

In [ ]:
# Reference date = latest transaction date + 1 day
reference_date = (
    transactions_clean["transaction_timestamp"].max()
    + pd.Timedelta(days=1)
)

# Create RFM metrics for purchasing customers
rfm = (
    transactions_clean
    .groupby("customer_id")
    .agg(
        last_purchase_date=("transaction_timestamp", "max"),
        frequency=("transaction_id", "count"),
        monetary=("order_value", "sum")
    )
    .reset_index()
)

# Recency in days
rfm["recency"] = (
    reference_date - rfm["last_purchase_date"]
).dt.days

# Create quartile scores
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    4,
    labels=[4, 3, 2, 1],
    duplicates="drop"
).astype(int)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["M_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
).astype(int)

# Combined RFM score
rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

# Business-friendly customer segments
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 3 and f >= 3 and m >= 3:
        return "Champions"
    elif r >= 3 and f >= 2:
        return "Loyal Customers"
    elif r >= 3 and m >= 3:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and m >= 3:
        return "High Value At Risk"
    elif r <= 2 and f <= 2:
        return "Low Engagement"
    else:
        return "Developing Customers"

rfm["customer_segment"] = rfm.apply(
    assign_segment,
    axis=1
)


segment_summary = (
    rfm
    .groupby("customer_segment")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

print("RFM customers:", len(rfm))
print("\nCustomer segments:")
print(
    segment_summary.to_string(index=False)
)

In [ ]:
# Connect RFM segments to acquisition information
acquisition_rfm = customer_acquisition.merge(
    rfm[
        [
            "customer_id",
            "recency",
            "frequency",
            "monetary",
            "customer_segment"
        ]
    ],
    on="customer_id",
    how="inner"
)

channel_segments = (
    acquisition_rfm
    .groupby(
        ["acquisition_channel", "customer_segment"]
    )
    .agg(
        customers=("customer_id", "nunique"),
        total_revenue=("monetary", "sum"),
        avg_customer_revenue=("monetary", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_recency=("recency", "mean")
    )
    .reset_index()
)

# Add percentage of each channel's customers
channel_totals = (
    channel_segments
    .groupby("acquisition_channel")["customers"]
    .transform("sum")
)

channel_segments["customer_share"] = (
    channel_segments["customers"] / channel_totals
)

#sort
channel_segments = channel_segments.sort_values(
    ["acquisition_channel", "total_revenue"],
    ascending=[True, False]
)

print(channel_segments.to_string(index=False))

## 5. Time & Channel Analysis

In [ ]:
# MONTHLY ACQUISITION & REVENUE TRENDS

# Create acquisition month
customer_metrics["acquisition_month"] = (
    customer_metrics["acquisition_date"]
    .dt.to_period("M")
    .astype(str)
)

# Monthly performance
monthly_acquisition = (
    customer_metrics
    .groupby("acquisition_month")
    .agg(
        acquired_customers=("customer_id", "nunique"),
        purchasing_customers=("is_customer", "sum"),
        repeat_customers=("is_repeat_customer", "sum"),
        total_revenue=("total_revenue", "sum"),
        total_orders=("total_orders", "sum")
    )
    .reset_index()
)

# Derived metrics
monthly_acquisition["purchase_conversion_rate"] = (
    monthly_acquisition["purchasing_customers"]
    / monthly_acquisition["acquired_customers"]
)

monthly_acquisition["repeat_rate"] = (
    monthly_acquisition["repeat_customers"]
    / monthly_acquisition["purchasing_customers"]
)

monthly_acquisition["revenue_per_acquired_customer"] = (
    monthly_acquisition["total_revenue"]
    / monthly_acquisition["acquired_customers"]
)

monthly_acquisition["average_order_value"] = (
    monthly_acquisition["total_revenue"]
    / monthly_acquisition["total_orders"]
)

# Month-over-month revenue change
monthly_acquisition["revenue_mom_change"] = (
    monthly_acquisition["total_revenue"]
    .pct_change()
)

# Month-over-month acquisition change
monthly_acquisition["acquisition_mom_change"] = (
    monthly_acquisition["acquired_customers"]
    .pct_change()
)

print("Monthly records:", len(monthly_acquisition))

print("\nFirst 10 months:")
print(
    monthly_acquisition.head(10).to_string(index=False)
)

print("\nLast 10 months:")
print(
    monthly_acquisition.tail(10).to_string(index=False)
)

In [ ]:
# Monthly channel performance

customer_metrics["acquisition_month"] = (
    customer_metrics["acquisition_date"]
    .dt.to_period("M")
    .astype(str)
)

monthly_channel = (
    customer_metrics
    .groupby(
        [
            "acquisition_month",
            "acquisition_channel"
        ]
    )
    .agg(
        acquired_customers=("customer_id", "nunique"),
        purchasing_customers=("is_customer", "sum"),
        repeat_customers=("is_repeat_customer", "sum"),
        total_revenue=("total_revenue", "sum"),
        total_orders=("total_orders", "sum")
    )
    .reset_index()
)

monthly_channel["purchase_conversion_rate"] = (
    monthly_channel["purchasing_customers"]
    / monthly_channel["acquired_customers"]
)

monthly_channel["repeat_rate"] = (
    monthly_channel["repeat_customers"]
    / monthly_channel["purchasing_customers"]
)

monthly_channel["revenue_per_customer"] = (
    monthly_channel["total_revenue"]
    / monthly_channel["acquired_customers"]
)

monthly_channel["average_order_value"] = (
    monthly_channel["total_revenue"]
    / monthly_channel["total_orders"]
)

print("Shape:", monthly_channel.shape)

print("\nSample:")
print(
    monthly_channel.head(12).to_string(index=False)
)

## 6. Campaign Type & Budget Optimization

In [ ]:
campaign_type_performance = (
    campaign_performance
    .groupby("acquisition_campaign_type")
    .agg(
        campaigns=("acquisition_campaign", "nunique"),
        acquired_customers=("acquired_customers", "sum"),
        purchasing_customers=("purchasing_customers", "sum"),
        repeat_customers=("repeat_customers", "sum"),
        total_revenue=("total_revenue", "sum"),
        total_budget=("campaign_budget", "sum"),
        total_orders=("total_orders", "sum")
    )
    .reset_index()
)

# Derived metrics
campaign_type_performance["conversion_rate"] = (
    campaign_type_performance["purchasing_customers"]
    / campaign_type_performance["acquired_customers"]
)

campaign_type_performance["repeat_rate"] = (
    campaign_type_performance["repeat_customers"]
    / campaign_type_performance["purchasing_customers"]
)

campaign_type_performance["revenue_per_customer"] = (
    campaign_type_performance["total_revenue"]
    / campaign_type_performance["acquired_customers"]
)

campaign_type_performance["budget_based_cac"] = (
    campaign_type_performance["total_budget"]
    / campaign_type_performance["acquired_customers"]
)

campaign_type_performance["budget_based_roas"] = (
    campaign_type_performance["total_revenue"]
    / campaign_type_performance["total_budget"]
)

campaign_type_performance["average_order_value"] = (
    campaign_type_performance["total_revenue"]
    / campaign_type_performance["total_orders"]
)

# Sort by ROAS
campaign_type_performance = campaign_type_performance.sort_values(
    "budget_based_roas",
    ascending=False
)

print(
    campaign_type_performance.to_string(index=False)
)

In [ ]:
# Budget allocation recommendation

budget_recommendation = campaign_type_performance.copy()

# Normalize performance metrics to 0-1
def min_max_score(series):
    if series.max() == series.min():
        return pd.Series(1.0, index=series.index)
    return (
        (series - series.min())
        / (series.max() - series.min())
    )

# Higher is better
budget_recommendation["roas_score"] = min_max_score(
    budget_recommendation["budget_based_roas"]
)

budget_recommendation["revenue_score"] = min_max_score(
    budget_recommendation["revenue_per_customer"]
)

budget_recommendation["repeat_score"] = min_max_score(
    budget_recommendation["repeat_rate"]
)

# Lower CAC is better
budget_recommendation["cac_score"] = (
    1 - min_max_score(
        budget_recommendation["budget_based_cac"]
    )
)

# Weighted overall score
budget_recommendation["overall_score"] = (
    0.40 * budget_recommendation["roas_score"]
    + 0.25 * budget_recommendation["cac_score"]
    + 0.20 * budget_recommendation["repeat_score"]
    + 0.15 * budget_recommendation["revenue_score"]
)

# Rank
budget_recommendation["recommendation_rank"] = (
    budget_recommendation["overall_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Recommended allocation
budget_recommendation["recommended_allocation_pct"] = (
    budget_recommendation["overall_score"]
    / budget_recommendation["overall_score"].sum()
    * 100
)

budget_recommendation = budget_recommendation.sort_values(
    "recommendation_rank"
)

print(
    budget_recommendation[
        [
            "acquisition_campaign_type",
            "budget_based_roas",
            "budget_based_cac",
            "repeat_rate",
            "revenue_per_customer",
            "overall_score",
            "recommendation_rank",
            "recommended_allocation_pct"
        ]
    ].to_string(index=False)
)

print(
    "\nAllocation total:",
    round(
        budget_recommendation["recommended_allocation_pct"].sum(),
        2
    ),
    "%"
)

## 7. SQL Business Analysis

### SQL Database Setup

The cleaned analytical datasets were loaded into a SQLite database to perform
business-oriented SQL analysis using aggregations, CTEs, subqueries and
window functions.

In [ ]:
import sqlite3

conn = sqlite3.connect("marketing_analytics.db")

print("SQLite database connected successfully.")

In [ ]:
sql_tables = {
    "customer_metrics": "../outputs/customer_metrics.csv",
    "campaign_performance": "../outputs/campaign_performance.csv",
    "channel_performance": "../outputs/channel_performance.csv",
    "monthly_acquisition": "../outputs/monthly_acquisition.csv",
    "monthly_channel": "../outputs/monthly_channel.csv",
    "rfm_customer_segments": "../outputs/rfm_customer_segments.csv",
    "segment_summary": "../outputs/segment_summary.csv",
    "campaign_type_performance": "../outputs/campaign_type_performance.csv",
    "budget_recommendation": "../outputs/budget_recommendation.csv",
    "marketing_funnel": "../outputs/marketing_funnel.csv"
}

In [ ]:
for table_name, file_path in sql_tables.items():
    df = pd.read_csv(file_path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"{table_name}: {len(df):,} rows")

print("\nAll analytical tables loaded.")